<!-- COMMONS LAUNCHER v3 · generated by tools/notebooks.py · do not edit by hand -->
<a href="https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials"><img src="https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/brand/synapsa-commons-badge.png" alt="Synapsa Commons" height="36"></a>

Free, hands-on AI courses that run anywhere, from the team building Synapsa, an AI-native
learning platform.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/lessons/T03-L01-bpe-from-scratch/lesson.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/lessons/T03-L01-bpe-from-scratch/lesson.ipynb)
[![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master?labpath=lessons/T03-L01-bpe-from-scratch/lesson.ipynb)
[![Open in Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials)

This lesson needs Python 3.11 or newer with numpy and matplotlib, which Colab, Kaggle,
Binder and Codespaces already have. The cell below installs `tokenizers==0.23.2`, and does
nothing where they are already present. On Kaggle, switch Internet on in the notebook's
settings first; Kaggle allows that only for phone-verified accounts.

In [ ]:
# --- COMMONS LAUNCHER v3 · generated by tools/notebooks.py · do not edit by hand ---
# Makes this notebook run anywhere. Every line is a no-op when the thing is already present,
# so a local clone pays nothing and an online notebook repairs itself.
import importlib.util, os, subprocess, sys, urllib.request
from pathlib import Path

COMMONS_PIP = [("tokenizers", "tokenizers==0.23.2")]            # (import name, pinned pip spec) for what this lesson imports
COMMONS_SIBLINGS = []    # files that must sit beside the notebook
# A fork, a classroom mirror or an offline copy can serve the files from elsewhere by setting
# COMMONS_RAW_OVERRIDE before running this cell.
COMMONS_RAW = os.environ.get("COMMONS_RAW_OVERRIDE") or "https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/lessons/T03-L01-bpe-from-scratch/"

# Resolve siblings against the LESSON's own directory, not the working directory. A notebook
# has no __file__ and runs with cwd alongside itself; a grader imports this file from the repo
# root. Checking cwd blindly makes the grader think every sibling is missing and reach for the
# network -- which would put a download on a graded path.
try:
    COMMONS_DIR = Path(__file__).resolve().parent
except NameError:
    COMMONS_DIR = Path.cwd()


def commons_host() -> str:
    """Name the notebook service we are on. Used for the message, and for honest errors."""
    try:
        if importlib.util.find_spec("google.colab") is not None:
            return "Google Colab"
    except (ImportError, ValueError):
        pass
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "Kaggle"
    if os.environ.get("BINDER_SERVICE_HOST"):
        return "Binder"
    if os.environ.get("CODESPACES"):
        return "GitHub Codespaces"
    return "a local Python environment"


_missing = [pip for imp, pip in COMMONS_PIP if importlib.util.find_spec(imp) is None]
if _missing:
    print("installing " + ", ".join(_missing) + " ...")
    # pip everywhere a student is likely to be; uv-managed local venvs ship without pip.
    if importlib.util.find_spec("pip") is not None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)
    else:
        subprocess.run(["uv", "pip", "install", "-q", "--python", sys.executable, *_missing],
                       check=True)
    importlib.invalidate_caches()

_fetched = []
for _name in COMMONS_SIBLINGS:
    if not (COMMONS_DIR / _name).exists():
        (COMMONS_DIR / _name).parent.mkdir(parents=True, exist_ok=True)
        try:
            urllib.request.urlretrieve(COMMONS_RAW + _name, COMMONS_DIR / _name)
            _fetched.append(_name)
        except Exception as _e:  # Kaggle disables the internet by default; say so plainly
            raise RuntimeError(
                f"this lesson needs {_name} beside the notebook and could not fetch it "
                f"({_e}). On Kaggle, switch Internet on in the notebook settings panel "
                f"(Kaggle allows that only for phone-verified accounts); otherwise download it "
                f"from {COMMONS_RAW + _name} and upload it beside the notebook."
            ) from None

print("ready on " + commons_host() + ("; fetched " + ", ".join(_fetched) if _fetched else ""))
# --- END COMMONS LAUNCHER ---

# T03-L01 · Byte-level BPE from scratch, and the token tax

**You will build:** a complete byte-pair-encoding tokenizer — trainer, encoder and decoder —
in about forty lines of standard-library Python, and then a measurement of what it charges
the same sentence in five writing systems.

**Time:** ~60 minutes · **Runs on:** a laptop CPU, 8 GiB RAM, no GPU, no download
· **Prerequisites:** `T00-L01-the-8gb-track`

By the end you will be able to:

1. Implement pair counting, a deterministic argmax, a non-overlapping merge, and the
   training loop that drives them.
2. Implement `encode` and `decode`, and prove losslessness with a round-trip property test
   across five scripts.
3. Measure tokens, characters and UTF-8 bytes for one sentence in Latin, Cyrillic,
   Devanagari, Arabic and Han script, and report the penalty each pays.
4. Explain why starting from bytes removes the unknown token, and what it costs instead.
5. Show that the penalty is a property of the training corpus, not of the algorithm — by
   retraining the identical code and watching it move.

Nothing is downloaded. The corpus and every sample sentence are text embedded in this file.

In [ ]:
# Setup: everything the lesson needs, in one cell, with versions printed.
import contextlib
import io
import sys
import time
import traceback
from typing import Callable, NamedTuple

_LESSON_T0 = time.perf_counter()
print("python", sys.version.split()[0])

# The constants that define this tokenizer. T03-L02 rebuilds this trainer in C++ and has to
# reproduce your merge list exactly, so these values — and the tie-break in exercise 3 — are
# part of the specification, not preferences.
BOUNDARY = 256          # the symbol between two words. Never merged across.
FIRST_MERGE_ID = 257    # the k-th merge is given id 257 + k
MIN_COUNT = 2           # a pair that occurs once buys no compression
VOCAB_SIZE = 768        # the budget every table in this lesson is measured at
_WHITESPACE = frozenset(b" \t\n\r\f\v")

_FAILED_CHECKS: list[str] = []
_STATUS: dict[str, str] = {}  # label -> "passed" | "failed" | "not started"

# The exercises, in the order you meet them. The progress board at the foot of the
# notebook is built from this, and a demo that is waiting on one names it from here.
_EXERCISES = {
    "exercise 1": "get_stats()",
    "exercise 2": "merge()",
    "exercise 3": "best_pair()",
    "exercise 4": "train_bpe()",
    "exercise 5": "encode() and decode()",
    "exercise 6": "token_tax()",
}


def _try(label: str, check: Callable[[], None], needs: tuple[str, ...] = ()) -> None:
    """Run a check, or a demo that depends on your code, without derailing the notebook.

    A stub you have not filled in yet simply says so. A wrong answer prints the check's own
    message — which names the likely mistake — and the notebook carries on, so one broken
    exercise never hides the feedback on the other five.

    A demo names the exercises it `needs`. Until each of them has passed its check, the demo
    says which one it is waiting for and skips, rather than failing half-way through its
    output. Every outcome is recorded in `_STATUS`, which the progress board at the foot reads.
    """
    waiting = [name for name in needs if _STATUS.get(name) != "passed"]
    if waiting:
        _STATUS[label] = "not started"
        todo = ", ".join(f"{name} ({_EXERCISES[name]})" for name in waiting)
        print(f"{label}: skipped — this needs {todo} to pass first.")
        return
    try:
        check()
    except NotImplementedError as exc:
        _STATUS[label] = "not started"
        stub = traceback.extract_tb(exc.__traceback__)[-1].name
        print(f"{label}: not implemented yet — fill in {stub}() above, then re-run this cell.")
    except AssertionError as exc:
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: FAILED — {exc}")
    except Exception as exc:  # a half-finished implementation raising something else
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: raised {type(exc).__name__}: {exc}")
    else:
        _STATUS[label] = "passed"


_MARKS = {"passed": "✅", "failed": "❌", "not started": "⏳"}


def _progress_board() -> None:
    """One line per exercise, marked passed, failed or not started, then the tally."""
    width = max(len(what) for what in _EXERCISES.values())
    print("progress board")
    for label, what in _EXERCISES.items():
        state = _STATUS.get(label, "not started")
        print(f"  {_MARKS[state]} {label:<11} {what:<{width}}  {state}")
    done = sum(_STATUS.get(label) == "passed" for label in _EXERCISES)
    print(f"\n{done} of {len(_EXERCISES)} exercises complete")
    failing = [label for label, state in _STATUS.items() if state == "failed"]
    if failing:
        print("failing right now: " + ", ".join(failing) + ". Each one printed what went wrong "
              "in its own cell above, and every exercise heading has hints you can open.")
    elif done < len(_EXERCISES):
        print("work top to bottom: every exercise heading has hints you can open.")

## 1. The corpus and the samples

`ENGLISH_CORPUS` is original English prose written for this lesson, so it carries no
third-party licence and nothing is downloaded. It is small and repetitive in its function
words, which is what gives a few hundred merges something to find.

`SAMPLES` is one aphorism in five writing systems. They say roughly the same thing and are
**not** the same length, which is why every table below also reports a per-character figure.

In [ ]:
ENGLISH_CORPUS = """\
A tokenizer is the first thing a language model does and the last thing anyone thinks about.
It takes the text you wrote and turns it into a list of integers, and the model never sees anything else.
The model does not read characters, and the model does not read words. The model reads tokens.
Everything the model knows about your text arrives through that list of integers.
A tokenizer that splits your text badly will cost you money on every request you ever make.
The tokenizer is trained once and then frozen, and every user of the model lives with that decision.
Training a tokenizer means choosing which sequences of bytes deserve a number of their own.
The sequences that appear often in the training text get short names, and the rest are spelled out.
This is the whole idea, and the rest of the work is bookkeeping.
Byte pair encoding starts from the raw bytes and invents a new symbol for the commonest adjacent pair.
It counts every adjacent pair in the text, picks the pair that occurs most often, and replaces it everywhere.
Then it counts again, picks again, and replaces again, until it has made as many merges as you asked for.
Each merge adds one symbol to the vocabulary, and each merge makes the text a little shorter.
The merge list is the whole model. There is nothing else to save and nothing else to ship.
A vocabulary is just the merge list read from the top, and the order of the merges is what makes it reproducible.
If two pairs occur the same number of times, something has to break the tie, and that rule is part of the model.
Break the tie differently and you get a different vocabulary from the same text, which is a bug you will find much later.
Starting from bytes rather than from characters is what makes the whole thing safe.
There are only two hundred and fifty six byte values, and every possible file is made of them.
A vocabulary that starts from bytes can therefore spell any text at all, including text it has never seen.
No character can ever fall outside the vocabulary, so there is no need for an unknown token.
A tokenizer that starts from characters must decide what to do with a character it has never met.
Usually it replaces that character with a special symbol, and the information in it is simply gone.
The model cannot recover what the tokenizer threw away, because the model never sees the original text.
Starting from bytes means the worst case is a long list of tokens, not a lost one.
A long list of tokens is expensive. A lost token is wrong. Those are very different failures.
The cost of a tokenizer is not paid by the person who trained it.
It is paid by every user of the model, on every request, forever, in tokens.
Two people can write the same sentence and be charged very different amounts for it.
The difference is not in what they wrote but in which text the tokenizer happened to be trained on.
Text that looks like the training text gets short tokens, and text that does not gets long ones.
A word that appeared ten thousand times in the training text probably has a token of its own.
A word that never appeared at all is spelled out one byte at a time.
This is not a flaw in byte pair encoding. It is exactly what byte pair encoding was asked to do.
The flaw, if there is one, is in the text it was trained on, and in who chose that text.
A model with a fixed context window holds a fixed number of tokens, not a fixed amount of meaning.
If your language costs more tokens per sentence, you fit less of your document into the same window.
You pay more for the same request, you wait longer for the same answer, and you get less context.
None of those three costs is visible in the model weights, and all three are decided by the tokenizer.
Measuring this is easy and almost nobody does it. You count the tokens and you divide.
Count the characters, count the bytes, count the tokens, and put the three numbers side by side.
The ratio of tokens to characters is the number that travels between languages.
Raw token counts do not travel, because a translated sentence is never the same length as the original.
Dividing by characters removes most of that difference and leaves the part the tokenizer caused.
The encoding itself is the floor. UTF-8 spends one byte on an ASCII character and more on everything else.
A tokenizer that has learned nothing about a script can do no better than one token per byte.
That floor is the worst case, and a script the tokenizer has never seen sits exactly on it.
Every merge the tokenizer learned for a script lifts that script off the floor a little.
So the fix is not a better algorithm. The fix is training text that contains the languages you serve.
You can prove that to yourself in a minute by training the same algorithm on a different corpus.
Train it on English and measure five scripts, then train it on all five and measure again.
The algorithm did not change, the code did not change, and the numbers move a long way.
That is the whole argument, and it is worth being able to run rather than to quote.
A merge that never fires is a wasted slot in the vocabulary, and the vocabulary is a fixed size.
Every slot you spend on one language is a slot you did not spend on another.
This is why vocabulary size and training mixture are the two decisions that matter most.
The rest of the tokenizer is an implementation detail, and a fast one is not a fair one.
Speed is easy to measure and fairness is easy to ignore, so speed is what gets reported.
A tokenizer is a small program with a very long shadow, and the shadow falls unevenly.
The merge loop itself is about thirty lines of code and you can write it in an afternoon.
Reading those thirty lines is worth more than reading a paper about them.
Write the counter, write the merge, write the loop, and then measure what you built.
Measure it on text you did not write and in a script you do not read.
The number you get back is the number your users would have paid.
Do that once and you will never again look at a token count as a neutral fact.
The token count is a choice, and somebody made it on your behalf.
Now you know how to make it yourself, and how to check the one you were given.
"""

SAMPLES = {
    "Latin (English)":    "Knowledge is power.",
    "Cyrillic (Russian)": "Знание — сила.",
    "Devanagari (Hindi)": "ज्ञान ही शक्ति है।",
    "Arabic":             "المعرفة قوة.",
    "Han (Chinese)":      "知识就是力量。",
}

## 2. Bytes are not characters, and the difference is already a bill

Count the characters, then count the UTF-8 bytes. UTF-8 spends one byte on an ASCII
character and up to four on others (RFC 3629, in `claims.yaml`), so the byte counts diverge
before any tokenizer has run at all. Run it — every number is measured from the strings.

In [ ]:
if __name__ == "__main__":
    print(f"{'script':22} {'chars':>6} {'utf-8 bytes':>12} {'bytes/char':>11}")
    for _name, _s in SAMPLES.items():
        _b = len(_s.encode("utf-8"))
        print(f"{_name:22} {len(_s):6d} {_b:12d} {_b / len(_s):11.2f}")
    print("\nThat last column is the encoding alone: no vocabulary, no training data, no")
    print("model. It is the floor, and by section 10 you will have measured who sits on it.")

## 3. Why start from bytes: the unknown token, and what replaces it

A tokenizer built over *characters* must decide what to do when it meets a character its
vocabulary does not contain. The usual answer is a special "unknown" symbol, and the
information in that character is then gone — the model never sees the original text, so it
cannot recover what the tokenizer discarded.

A tokenizer built over *bytes* cannot have this problem. There are only 256 byte values and
every file is made of them, so the alphabet is complete by construction. GPT-2's own encoder
gives exactly this reason for going byte-level (`claims.yaml`). The cost is real, though, and
it is the subject of this lesson: the worst case for a byte-level tokenizer is a *long* list
of tokens, never a lost one.

In [ ]:
def character_vocabulary_demo(train_text: str, test_text: str) -> dict:
    """Build a character vocabulary from `train_text`, then try to represent `test_text`.

    This is the tokenizer this lesson is NOT building, shown so the failure is concrete.
    """
    vocab = set(train_text)
    lost = [c for c in test_text if c not in vocab]
    return {
        "recovered": "".join(c if c in vocab else "�" for c in test_text),
        "characters_lost": len(lost),
        "fraction_lost": len(lost) / len(test_text),
        "byte_level_recovered": test_text,          # bytes can always spell it
        "byte_level_tokens": len(test_text.encode("utf-8")),
    }


if __name__ == "__main__":
    _demo = character_vocabulary_demo(SAMPLES["Latin (English)"], SAMPLES["Han (Chinese)"])
    print("a character vocabulary trained on English, asked to represent Chinese:")
    print(f"  recovered        {_demo['recovered']!r}")
    print(f"  characters lost  {_demo['characters_lost']} of "
          f"{len(SAMPLES['Han (Chinese)'])}  ({_demo['fraction_lost']:.0%})")
    print("the same text through a byte-level vocabulary:")
    print(f"  recovered        {_demo['byte_level_recovered']!r}")
    print(f"  cost             {_demo['byte_level_tokens']} tokens, and nothing is lost")
    print("\nExpensive is not the same as wrong — but by section 10 you will have measured")
    print("how expensive, and for whom.")

## 4. The algorithm, and the rule that makes it reproducible

BPE starts from bytes and invents a new symbol for the commonest adjacent pair, over and
over. Sennrich, Haddow and Birch brought it to neural translation from compression
(`claims.yaml`); every modern tokenizer still runs this loop.

Pre-tokenisation comes first, and it is given to you. Words are split on whitespace, the
whitespace itself is dropped, and a `BOUNDARY` symbol is placed after each word. Pairs are
never counted across a boundary, so no token can ever span two words.

In [ ]:
def pretokenise(blob: bytes) -> list[int]:
    """Bytes of a word, then BOUNDARY, then the next word. Whitespace itself is dropped.

    Given to you, and identical to the C++ version in T03-L02: the two trainers must agree.

    Example:
        >>> pretokenise(b"hi yo")
        [104, 105, 256, 121, 111, 256]
    """
    seq, in_word = [], False
    for byte in blob:
        if byte in _WHITESPACE:
            if in_word:
                seq.append(BOUNDARY)
                in_word = False
        else:
            seq.append(byte)
            in_word = True
    if in_word:
        seq.append(BOUNDARY)
    return seq


if __name__ == "__main__":
    print(pretokenise(b"the cat"), "  <- 256 is the word boundary")

## 5. Exercise 1 — `get_stats`

Count every adjacent pair in one left-to-right scan. Two rules: a pair that touches
`BOUNDARY` on either side is not counted at all, and counting **is** allowed to overlap —
three identical symbols in a row hold two pairs. (Only *merging* is non-overlapping, which
is exercise 2.)

<details><summary>💡 Hint 1 — what to think about</summary>

Counting is not merging. When the pair at position i is counted, does that stop you
looking at the pair that starts at i + 1? And a pair that touches `BOUNDARY`: should it
appear in the dict with a count of zero, or not appear at all?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Visit every index from the first symbol to the one before the last, take the pair that
starts there, skip it if either side is `BOUNDARY`, and otherwise add one to its count.
Move one position at a time, always. A range that stops one short of the end also looks
after empty and one-symbol sequences, with no special case.
</details>

In [ ]:
def get_stats(seq: list[int]) -> dict[tuple[int, int], int]:
    """Count adjacent pairs in `seq`, skipping any pair that touches BOUNDARY.

    One O(n) scan. Return a dict mapping (left, right) to how many times it occurs.

    Example:
        >>> get_stats([97, 98, 97, 98, 256])
        {(97, 98): 2, (98, 97): 1}
        >>> get_stats([7, 7, 7])
        {(7, 7): 2}
    """
    # YOUR CODE HERE
    raise NotImplementedError


# Public checks — run these as often as you like.
def _check_get_stats() -> None:
    got = get_stats(pretokenise(b"ab ab"))
    assert got == {(97, 98): 2}, (
        f"get_stats on 'ab ab' gave {got}, expected {{(97, 98): 2}} — any entry mentioning "
        f"{BOUNDARY} means you are counting across the word boundary; skip the pair entirely "
        "rather than recording it as zero."
    )
    assert get_stats([7, 7, 7]) == {(7, 7): 2}, (
        "[7,7,7] holds TWO (7,7) pairs — counting overlaps. If you got 1 you are skipping "
        "ahead by two after a match, which is the rule for merging, not for counting."
    )
    assert get_stats([]) == {} and get_stats([5]) == {}, (
        "a sequence shorter than two symbols has no pairs — guard the loop instead of "
        "indexing seq[i+1] past the end."
    )
    assert get_stats([BOUNDARY, BOUNDARY, 4]) == {}, (
        "two boundaries in a row still yield no pair, and neither does a boundary followed "
        "by a symbol."
    )
    big = get_stats(pretokenise(ENGLISH_CORPUS.encode("utf-8")))
    assert big, "no pairs at all on real text — is the function returning before the loop?"
    print(f"exercise 1 looks right: {len(big)} distinct pairs in the corpus, "
          f"commonest occurs {max(big.values())} times")

In [ ]:
if __name__ == "__main__":
    _try("exercise 1", _check_get_stats)

## 6. Exercise 2 — `merge`

Replace every **non-overlapping, left-to-right** occurrence of one pair with a new id. After
a match the cursor advances by two: the second symbol has been consumed and may not be read
again. `[7, 7, 7]` merging `(7,7)` therefore becomes `[300, 7]`, not `[300, 300]`.

<details><summary>💡 Hint 1 — what to think about</summary>

Where does the scan resume after a match? The second symbol of the pair has been used
up, and reading it again is the commonest wrong answer in this lesson. Then look at the
other end: can the very last two symbols still match?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Keep an explicit read cursor and an output list, and loop while the cursor is inside the
sequence. If there is a next symbol and the two equal the pair, emit the new id and move
the cursor on by two; otherwise copy the current symbol across and move on by one.
</details>

In [ ]:
def merge(seq: list[int], pair: tuple[int, int], new_id: int) -> list[int]:
    """Replace every non-overlapping left-to-right occurrence of `pair` with `new_id`.

    Symbols that are not part of a match are copied through unchanged, in order.

    Example:
        >>> merge([7, 7, 7], (7, 7), 300)
        [300, 7]
        >>> merge([5, 1, 2, 1, 2], (1, 2), 300)
        [5, 300, 300]
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_merge() -> None:
    triple = merge([7, 7, 7], (7, 7), 300)
    assert triple == [300, 7], (
        f"[7,7,7] merged on (7,7) must be [300,7], got {triple} — after a match advance the "
        "cursor by TWO; advancing by one consumes the middle symbol twice."
    )
    assert merge([1, 2, 3], (8, 9), 300) == [1, 2, 3], (
        "a pair that never occurs must leave the sequence untouched, same length and all."
    )
    assert merge([1, 2, 1, 2], (1, 2), 300) == [300, 300], (
        "two separate occurrences must both merge — do not stop at the first match."
    )
    assert merge([5, 1, 2], (1, 2), 300) == [5, 300], (
        "symbols before a match must be copied through, in order."
    )
    assert merge([1, 2], (1, 2), 300) == [300], (
        "a match at the very end must still fire — the guard is i + 1 < len(seq), and it "
        "protects the lookahead; it does not exclude the final pair."
    )
    assert merge([300, 301, 300, 301], (300, 301), 500) == [500, 500], (
        "ids above 255 must merge like any other: symbols are ints, not bytes."
    )
    print("exercise 2 looks right: overlap, tail, absent pair and multi-byte ids all handled")

In [ ]:
if __name__ == "__main__":
    _try("exercise 2", _check_merge)

## 7. Exercise 3 — `best_pair`, and why the tie-break is the model

Highest count wins. When two pairs tie, **the smaller `(a, b)` tuple wins** — and that rule
is not a detail. Whichever pair you return becomes token 257, which renames every merge
after it. A tie broken by dictionary order is a tokenizer that produces a different
vocabulary on a different machine, and the vocabulary is the artefact you ship.

<details><summary>💡 Hint 1 — what to think about</summary>

Your answer must not depend on the order the dict hands you its items: the same counts
inserted in a different order have to produce the same winner. What decides between two
pairs with equal counts, and what should a caller get back when there is nothing to
choose from?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Return `None` at once for an empty dict. Otherwise keep a running best, and replace it
only when a candidate's count is strictly higher, or the counts are equal and the
candidate pair compares smaller. Return the winner's two symbols and its count as one
flat tuple.
</details>

In [ ]:
def best_pair(stats: dict[tuple[int, int], int]) -> tuple[int, int, int] | None:
    """Return `(a, b, count)` for the winning pair, or `None` if `stats` is empty.

    Highest count wins. Ties go to the smaller `(a, b)` tuple — Python compares tuples
    exactly the way this rule needs, so `pair < best` is the whole tie-break.

    Example:
        >>> best_pair({(1, 2): 3, (4, 5): 9})
        (4, 5, 9)
        >>> best_pair({(9, 9): 4, (2, 7): 4})
        (2, 7, 4)
        >>> best_pair({}) is None
        True
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_best_pair() -> None:
    assert best_pair({(1, 2): 3, (4, 5): 9, (6, 7): 5}) == (4, 5, 9), (
        "the highest count must win outright when nothing ties — check the comparison is > "
        "and not >=, which lets a later equal-count pair displace the leader."
    )
    tie = best_pair({(9, 9): 4, (2, 7): 4})
    assert tie == (2, 7, 4), (
        f"a tie between (9,9) and (2,7) must go to (2,7), got {tie} — compare the tuples "
        "themselves; whichever one you happened to meet first is not a rule."
    )
    assert best_pair({}) is None, (
        "an empty stats dict must return None, not a pair of zeros — train_bpe relies on "
        "that signal to stop."
    )
    assert best_pair({(260, 261): 1}) == (260, 261, 1), "a single entry is its own winner"
    # The check the small cases cannot make: one tied set, four insertion orders. A real
    # tie-break answers identically every time; "the first maximum I met" follows the dict.
    tied = [(3, 4), (17, 18), (41, 42), (88, 89), (150, 151), (260, 261), (301, 302)]
    answers = {best_pair({p: 9 for p in order})
               for order in (tied, list(reversed(tied)), tied[3:] + tied[:3],
                             tied[1::2] + tied[0::2])}
    assert answers == {(3, 4, 9)}, (
        f"the same seven tied pairs produced {sorted(answers)} — the only thing that changed "
        "was insertion order, so you are returning the first maximum the dict handed you "
        "rather than the smallest (a, b). This is the case the examples above pass without "
        "any tie-break at all."
    )
    print("exercise 3 looks right: highest count wins, ties resolve to the smaller pair, and "
          "the answer does not depend on insertion order")

In [ ]:
if __name__ == "__main__":
    _try("exercise 3", _check_best_pair)

## 8. Exercise 4 — `train_bpe`

Now the loop: count, pick, record, apply, repeat. Three rules decide when it ends.

- The vocabulary already holds the 256 byte values plus `BOUNDARY`, so `vocab_size` buys
  `vocab_size - FIRST_MERGE_ID` merges. A `vocab_size` below `FIRST_MERGE_ID` is a `ValueError`.
- The k-th merge is given id `FIRST_MERGE_ID + k`.
- Stop early when the best pair occurs fewer than `MIN_COUNT` times — tested *before* the
  merge is recorded, because a pair seen once buys no compression.

<details><summary>💡 Hint 1 — what to think about</summary>

Three things decide how many rows `order` ends up with: the budget left after the byte
values and `BOUNDARY`, the stop rule, and whether each merge changes what the next count
sees. Which of them does a `vocab_size` of exactly `FIRST_MERGE_ID` exercise, and is
that an error?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Refuse a `vocab_size` below `FIRST_MERGE_ID` first. Pretokenise the UTF-8 bytes once,
then loop once per merge you can afford: count with your `get_stats`, choose with your
`best_pair`, and stop if there is no winner or its count is under `MIN_COUNT`. Only then
give it the next id, record it in `merges` and `order`, and replace the working sequence
with your `merge` of it. `build_vocab` turns the finished merges into `vocab`.
</details>

In [ ]:
class BPEModel(NamedTuple):
    """A trained tokenizer. The merge list is the whole model; `vocab` is derived from it."""

    merges: dict[tuple[int, int], int]        # (a, b) -> new id, in training order
    order: list[tuple[int, int, int, int]]    # (a, b, new_id, count) per merge, for inspection
    vocab: dict[int, bytes]                   # every id, expanded to the bytes it stands for


def build_vocab(merges: dict[tuple[int, int], int]) -> dict[int, bytes]:
    """Expand every id to the bytes it stands for. Given to you; `decode` needs it.

    Example:
        >>> build_vocab({(104, 105): 257})[257]
        b'hi'
    """
    vocab: dict[int, bytes] = {i: bytes([i]) for i in range(256)}
    vocab[BOUNDARY] = b""
    for (a, b), new_id in merges.items():
        vocab[new_id] = vocab[a] + vocab[b]
    return vocab


def train_bpe(text: str, vocab_size: int) -> BPEModel:
    """Train byte-level BPE on `text` and return the trained model.

    Steps: encode to UTF-8, `pretokenise`, then loop `vocab_size - FIRST_MERGE_ID` times —
    `get_stats`, `best_pair`, stop if there is no winner or its count is below `MIN_COUNT`,
    otherwise record `(a, b, new_id, count)` and apply the merge before counting again.
    Build `vocab` with `build_vocab` once the loop ends.

    Raise `ValueError` if `vocab_size` is below `FIRST_MERGE_ID`.

    Example:
        >>> m = train_bpe("ab ab ab", 258)
        >>> m.order
        [(97, 98, 257, 3)]
        >>> m.vocab[257]
        b'ab'
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_train_bpe() -> None:
    tiny = train_bpe("ab ab ab", 258)
    assert tiny.order == [(97, 98, 257, 3)], (
        f"train_bpe('ab ab ab', 258).order gave {tiny.order}, expected [(97, 98, 257, 3)] — "
        f"one merge, numbered {FIRST_MERGE_ID}, recording the count it was chosen on."
    )
    assert tiny.vocab[257] == b"ab", (
        "vocab[257] should expand to b'ab' — call build_vocab(merges) after the loop."
    )
    three = train_bpe("abcabcabc abcabcabc", 259)
    assert [row[:3] for row in three.order] == [(97, 98, 257), (257, 99, 258)], (
        f"expected the second merge to consume the first (ids {FIRST_MERGE_ID} then "
        f"{FIRST_MERGE_ID + 1}), got {[row[:3] for row in three.order]} — apply each merge to "
        "the sequence BEFORE counting again, or every merge is chosen from the same counts."
    )
    assert train_bpe("abcdef ghijkl", 300).order == [], (
        "no pair occurs twice here, so no merge is worth making however many are asked for — "
        f"stop when the winning count is below MIN_COUNT ({MIN_COUNT}), and test that before "
        "recording the merge."
    )
    capped = train_bpe(ENGLISH_CORPUS, 300)
    assert len(capped.order) == 300 - FIRST_MERGE_ID, (
        f"a vocab_size of 300 buys exactly {300 - FIRST_MERGE_ID} merges on a corpus this "
        f"size, got {len(capped.order)} — the 256 byte values and BOUNDARY are already "
        "spoken for."
    )
    assert train_bpe(ENGLISH_CORPUS, FIRST_MERGE_ID).order == [], (
        f"a vocab_size of exactly {FIRST_MERGE_ID} buys zero merges — that is a valid, empty "
        "model, not an error."
    )
    try:
        train_bpe("anything", 256)
    except ValueError:
        pass
    else:
        raise AssertionError(
            f"vocab_size below {FIRST_MERGE_ID} must raise ValueError — there is no room for "
            "a single merge, and silently training zero of them hides the caller's mistake."
        )
    print("exercise 4 looks right: ids numbered from 257, merges applied before recounting, "
          "stop rule and vocab_size guard all behaving")

In [ ]:
if __name__ == "__main__":
    _try("exercise 4", _check_train_bpe, needs=("exercise 1", "exercise 2", "exercise 3"))

## 9. Exercise 5 — `encode`, `decode`, and the round-trip property

Training produced a merge list. Encoding replays it: take a word's bytes and apply every
merge **in training order**. Whitespace bytes pass through untouched — no merge can contain
one, because pre-tokenisation dropped them all before counting.

Decoding expands each id through `vocab` and joins the bytes. The property that matters is
`decode(encode(s)) == s`, **for every string**, including scripts the corpus never contained.
That is what byte-level buys you, and you are about to test it.

<details><summary>💡 Hint 1 — what to think about</summary>

Pre-tokenisation threw the whitespace away; encoding must not, or `decode` has nothing
to put back. So as you walk the bytes you need to know, at every step, whether you are
inside a word. And at encode time, what is `BOUNDARY` for — anything at all?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Walk the UTF-8 bytes, gathering non-whitespace bytes into the current word. At a
whitespace byte, finish the word — every merge, in `model.merges` order, through your
`merge`, reassigning each time — emit its ids, then emit the whitespace byte itself.
Finish the last word after the loop. `decode` joins the `bytes` each id stands for in
`model.vocab` and decodes the whole once, replacing a broken character.
</details>

In [ ]:
def encode(text: str, model: BPEModel) -> list[int]:
    """Encode `text` to a list of token ids, applying `model.merges` in training order.

    Split the UTF-8 bytes into runs: a run of non-whitespace bytes is a word, and each
    whitespace byte is emitted as its own id. For each word, apply every merge in
    `model.merges` order with your own `merge` function. BOUNDARY is a training-time device
    and must never appear in the output.

    Example:
        >>> m = train_bpe("ab ab ab", 258)
        >>> encode("ab", m)
        [257]
        >>> encode("ab ab", m)
        [257, 32, 257]
    """
    # YOUR CODE HERE
    raise NotImplementedError


def decode(ids: list[int], model: BPEModel) -> str:
    """Expand `ids` back to text through `model.vocab`.

    Join the bytes each id stands for, then decode the result as UTF-8 with
    `errors="replace"` — a partial token can end mid-character, and that must not raise.

    Example:
        >>> m = train_bpe("ab ab ab", 258)
        >>> decode([257, 32, 257], m)
        'ab ab'
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_encode_decode() -> None:
    toy = train_bpe("ab ab ab", 258)
    assert encode("ab", toy) == [257], (
        "the single merge of this toy model should turn 'ab' into one token — are you "
        "applying model.merges to each word?"
    )
    spaced = encode("ab ab", toy)
    assert spaced == [257, 32, 257], (
        f"'ab ab' should encode to [257, 32, 257], got {spaced} — the space is emitted as its "
        f"own byte id (32), and {BOUNDARY} must never reach the output."
    )
    model = train_bpe(ENGLISH_CORPUS, VOCAB_SIZE)
    assert BOUNDARY not in encode(ENGLISH_CORPUS[:400], model), (
        f"{BOUNDARY} appeared in encode's output — it separates words during TRAINING only. "
        "Emit the whitespace bytes themselves, so decode can put them back."
    )
    for name, sample in SAMPLES.items():
        got = decode(encode(sample, model), model)
        assert got == sample, (
            f"round trip failed for {name}: encode then decode gave {got!r}, expected "
            f"{sample!r}. This model was trained on English only, which is exactly why it "
            "must still work — every byte value is in the vocabulary. A mismatch here usually "
            "means whitespace was dropped rather than emitted, or decode joined str instead "
            "of bytes."
        )
    assert decode(encode("", model), model) == "", "the empty string round-trips to itself"
    english = encode(SAMPLES["Latin (English)"], model)
    english_bytes = len(SAMPLES["Latin (English)"].encode("utf-8"))
    assert len(english) < english_bytes, (
        f"English encoded to {len(english)} tokens, no shorter than its {english_bytes} "
        "bytes — the merges are not being applied. Check you are iterating model.merges and "
        "reassigning the result of merge()."
    )
    print(f"exercise 5 looks right: all five scripts round-trip exactly, and English "
          f"compresses to {len(english)} tokens from {english_bytes} bytes")

In [ ]:
if __name__ == "__main__":
    _try("exercise 5", _check_encode_decode, needs=("exercise 2", "exercise 4"))

## 10. Exercise 6 — measure the tax

Now the measurement this lesson exists for. For each sample: characters, UTF-8 bytes, tokens,
and tokens per character. Raw token counts do not travel between languages — a translation is
never the same length as its original — so the per-character column is the one that compares.

<details><summary>💡 Hint 1 — what to think about</summary>

Three counts per sample and two ratios. Only one denominator lets a Hindi row be read
against an English one: which, and why does the other one flatten the whole table
towards the same figure?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Loop over the samples in the order given and build one dict per sample with exactly the
seven documented keys: the name, the text, its length, the length of its UTF-8 encoding,
the length of your `encode` of it, and the tokens and the bytes each divided by the
character count.
</details>

In [ ]:
def token_tax(model: BPEModel, samples: dict[str, str]) -> list[dict]:
    """One row per sample, in the order `samples` gives them.

    Each row is a dict with exactly these keys:
      "script", "text", "chars", "utf8_bytes", "tokens", "tokens_per_char", "bytes_per_char"

    Every number is computed, none is looked up. `tokens` is `len(encode(text, model))`.

    Example:
        >>> m = train_bpe("ab ab ab", 258)
        >>> row = token_tax(m, {"toy": "ab"})[0]
        >>> row["script"], row["chars"], row["tokens"]
        ('toy', 2, 1)
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_token_tax() -> None:
    model = train_bpe(ENGLISH_CORPUS, VOCAB_SIZE)
    rows = token_tax(model, SAMPLES)
    assert [r["script"] for r in rows] == list(SAMPLES), (
        "token_tax must return one row per sample, in the order they were given."
    )
    required = {"script", "text", "chars", "utf8_bytes", "tokens",
                "tokens_per_char", "bytes_per_char"}
    assert set(rows[0]) == required, (
        f"row keys are {sorted(rows[0])}, expected exactly {sorted(required)}"
    )
    latin = rows[0]
    assert latin["chars"] == len(SAMPLES["Latin (English)"]), "chars is len(text)"
    assert latin["utf8_bytes"] == len(SAMPLES["Latin (English)"].encode("utf-8")), (
        "utf8_bytes is len(text.encode('utf-8')), not len(text)"
    )
    assert abs(latin["tokens_per_char"] - latin["tokens"] / latin["chars"]) < 1e-9, (
        "tokens_per_char is tokens divided by CHARACTERS, not by bytes — dividing by bytes "
        "hides the very effect this table exists to show."
    )
    assert abs(latin["bytes_per_char"] - latin["utf8_bytes"] / latin["chars"]) < 1e-9, (
        "bytes_per_char is utf8_bytes divided by chars"
    )
    print(f"exercise 6 looks right: {len(rows)} rows measured against a "
          f"{len(model.order)}-merge vocabulary")

In [ ]:
if __name__ == "__main__":
    _try("exercise 6", _check_token_tax, needs=("exercise 4", "exercise 5"))

Now the table. Stare at the `tokens==bytes` column: where it says `yes`, the tokenizer
learned nothing whatsoever about that script and has fallen back to spelling it out one byte
at a time. That is the floor from section 2, and it is where most of the world's writing
systems sit in a vocabulary trained on English.

In [ ]:
def print_tax_table(rows: list[dict], baseline: str = "Latin (English)") -> None:
    """Print the measured table, with every figure relative to `baseline`. Given to you."""
    base = next(r for r in rows if r["script"] == baseline)["tokens_per_char"]
    print(f"{'script':22} {'chars':>6} {'bytes':>6} {'tokens':>7} {'tok/char':>9} "
          f"{'vs base':>8} {'tokens==bytes':>14}")
    for r in rows:
        print(f"{r['script']:22} {r['chars']:6d} {r['utf8_bytes']:6d} {r['tokens']:7d} "
              f"{r['tokens_per_char']:9.2f} {r['tokens_per_char'] / base:7.2f}x "
              f"{('yes' if r['tokens'] == r['utf8_bytes'] else 'no'):>14}")


def _show_tax() -> None:
    model = train_bpe(ENGLISH_CORPUS, VOCAB_SIZE)
    rows = token_tax(model, SAMPLES)
    print(f"vocabulary: {len(model.order)} merges trained on {len(ENGLISH_CORPUS)} "
          f"characters of English\n")
    print_tax_table(rows)
    base = rows[0]["tokens_per_char"]
    worst = max(rows, key=lambda r: r["tokens_per_char"])
    print(f"\nworst penalty: {worst['script']} at "
          f"{worst['tokens_per_char'] / base:.2f}x the English rate per character.")
    print("Commercial APIs meter usage in tokens and state context limits in tokens")
    print("(`claims.yaml`), so that multiplier is a price, a latency, and a shorter memory.")

In [ ]:
if __name__ == "__main__":
    _try("the tax table", _show_tax, needs=("exercise 4", "exercise 6"))

## 11. Check your trainer against a real one

`tokenizers` is the library most published BPE vocabularies are trained with, and its speed
comes from a Rust implementation. Its `ByteLevel` pre-tokenizer starts from an alphabet of
exactly 256 characters — one per byte value — the same starting point you used.

Train it on the same corpus and compare. The two are different programs with different
pre-tokenisers, so expect agreement within a token or two, not equality. What matters is that
the *shape* of the tax is the same in both.

In [ ]:
def reference_tokenizer_counts(corpus: str, vocab_size: int = VOCAB_SIZE) -> dict[str, int]:
    """Train HuggingFace's byte-level BPE on `corpus`; return tokens per sample. Given to you."""
    from tokenizers import Tokenizer, decoders, models, pre_tokenizers, trainers

    tok = Tokenizer(models.BPE())
    tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=True)
    tok.decoder = decoders.ByteLevel()
    trainer = trainers.BpeTrainer(vocab_size=vocab_size, show_progress=False,
                                  initial_alphabet=pre_tokenizers.ByteLevel.alphabet())
    tok.train_from_iterator([corpus], trainer)
    return {name: len(tok.encode(text).ids) for name, text in SAMPLES.items()}


def _show_reference_comparison() -> None:
    import tokenizers

    model = train_bpe(ENGLISH_CORPUS, VOCAB_SIZE)
    theirs = reference_tokenizer_counts(ENGLISH_CORPUS, VOCAB_SIZE)
    print(f"huggingface tokenizers {tokenizers.__version__} (Rust), same corpus, same budget\n")
    print(f"{'script':22} {'yours':>6} {'theirs':>7} {'difference':>11}")
    for name, text in SAMPLES.items():
        mine = len(encode(text, model))
        print(f"{name:22} {mine:6d} {theirs[name]:7d} {theirs[name] - mine:11d}")
    print("\nTwo independent implementations, the same corpus, the same verdict on every")
    print("script. The tax you measured is not an artefact of your code.")

In [ ]:
if __name__ == "__main__":
    _try("reference comparison", _show_reference_comparison,
         needs=("exercise 4", "exercise 5"))

## 12. The tax is the corpus, not the algorithm

It would be easy to read section 10 as a fact about Hindi or Chinese. It is not. Nothing in
the algorithm knows what a script is: it counts pairs and merges the commonest one. The
scripts that pay sit on the byte floor because **no merge ever fired for them** — and no
merge fired because their bytes were not in the corpus to be counted.

So change the corpus, change nothing else, and re-measure. The balanced corpus below is a
toy — the same handful of phrases repeated, enough to show the mechanism and nowhere near
enough to build a real vocabulary — but the code training on it is the code you wrote.

In [ ]:
PHRASEBOOK = {
    "Latin (English)":    ["Good morning.", "Thank you very much.", "The book is on the table."],
    "Cyrillic (Russian)": ["Доброе утро.", "Большое спасибо.", "Книга на столе."],
    "Devanagari (Hindi)": ["सुप्रभात।", "बहुत धन्यवाद।", "किताब मेज़ पर है।"],
    "Arabic":             ["صباح الخير.", "شكرا جزيلا.", "الكتاب على الطاولة."],
    "Han (Chinese)":      ["早上好。", "非常感谢。", "书在桌子上。"],
}


def balanced_corpus(repeats: int = 40) -> str:
    """English corpus plus every phrasebook script, repeated enough to be counted. Given to you."""
    parts = [ENGLISH_CORPUS]
    for script, phrases in PHRASEBOOK.items():
        block = " ".join(phrases + [SAMPLES[script]])
        parts.append((block + "\n") * repeats)
    return "\n".join(parts)


def _show_the_fix() -> None:
    english_only = train_bpe(ENGLISH_CORPUS, VOCAB_SIZE)
    balanced = train_bpe(balanced_corpus(), VOCAB_SIZE)
    before = {r["script"]: r for r in token_tax(english_only, SAMPLES)}
    after = {r["script"]: r for r in token_tax(balanced, SAMPLES)}
    base_before = before["Latin (English)"]["tokens_per_char"]
    base_after = after["Latin (English)"]["tokens_per_char"]
    print("same algorithm, same vocabulary budget, same sentences — only the corpus changed\n")
    print(f"{'script':22} {'tokens before':>14} {'tokens after':>13} "
          f"{'penalty before':>15} {'penalty after':>14}")
    for script in SAMPLES:
        b, a = before[script], after[script]
        print(f"{script:22} {b['tokens']:14d} {a['tokens']:13d} "
              f"{b['tokens_per_char'] / base_before:14.2f}x "
              f"{a['tokens_per_char'] / base_after:13.2f}x")
    print("\nNo line of the algorithm changed. The penalty is a decision about training data,")
    print("made once, by somebody else, and paid by everyone who uses the model afterwards.")

In [ ]:
if __name__ == "__main__":
    _try("the fix", _show_the_fix, needs=("exercise 4", "exercise 6"))

## 13. Common mistakes

- **Advancing by one after a match in `merge`.** `[7,7,7]` becomes `[300,300,7]`: the middle
  symbol is consumed twice, the sequence barely shrinks, and your merge list parts company
  with the C++ trainer in T03-L02. The cell below runs the bug rather than describing it.
- **Counting pairs across the boundary.** You invent tokens spanning two words, and the very
  first merge already disagrees with everyone else's.
- **Letting dictionary order break ties.** It passes today and produces a different
  vocabulary after the corpus grows by one word. The tie-break *is* the reproducibility
  guarantee.
- **Counting non-overlapping.** Counting and merging follow different rules: `[7,7,7]` holds
  two pairs but merges into one new symbol plus a leftover.
- **Recounting without applying the merge.** Every merge then comes from the same stale
  counts, and you record the same pair repeatedly under new ids.
- **Letting `BOUNDARY` escape into `encode`'s output.** It is a training-time device. Emit
  the whitespace bytes instead, or `decode` cannot put the spaces back.
- **Dividing tokens by bytes instead of characters.** It flattens the table to roughly 1.0
  everywhere and conceals exactly the effect you are measuring.
- **Reading the tax as a fact about a language.** It is a fact about a corpus, and section 12
  is the control experiment that shows it.

In [ ]:
def merge_advancing_by_one(seq: list[int], pair: tuple[int, int], new_id: int) -> list[int]:
    """`merge` with one character changed: `i += 2` became `i += 1`. The commonest wrong answer.

    Example:
        >>> merge_advancing_by_one([7, 7, 7], (7, 7), 300)
        [300, 300, 7]
    """
    a, b = pair
    out, i, n = [], 0, len(seq)
    while i < n:
        if i + 1 < n and seq[i] == a and seq[i + 1] == b:
            out.append(new_id)
            i += 1                      # the bug: the second symbol is read again
        else:
            out.append(seq[i])
            i += 1
    return out


def _show_the_bug() -> None:
    print(f"{'input':>20}  {'correct':<22} advance-by-one")
    for case in ([7, 7, 7], [7, 7, 7, 7], [7, 7, 7, 7, 7]):
        print(f"{str(case):>20}  {str(merge(case, (7, 7), 300)):<22}"
              f" {merge_advancing_by_one(case, (7, 7), 300)}")
    print("\nA run of n identical symbols should collapse to about n/2. The buggy column stays")
    print("nearly as long as the input, so the sequence stops shrinking and the merge list")
    print("diverges from the C++ trainer that has to reproduce it.")

In [ ]:
if __name__ == "__main__":
    _try("the advance-by-one bug", _show_the_bug, needs=("exercise 2",))

## 14. Self-check

1. Your `best_pair` returns the first maximum it meets while iterating `stats`, and every
   example in the notebook passes. What is actually wrong?
   - (a) nothing — Python dicts preserve insertion order, so it is deterministic
   - (b) it is slower than comparing tuples
   - (c) the winner depends on the order pairs were inserted, so the same corpus can produce
         different vocabularies — and the vocabulary is the artefact you ship
   - (d) it only matters when two pairs both have count zero

2. A vocabulary trained only on English round-trips a Chinese sentence exactly. Why?
   - (a) the trainer detected Chinese and added the characters it needed
   - (b) the alphabet is the 256 byte values, so every possible input can be spelled; the
         cost of an unseen script is a long token list, never a lost character
   - (c) UTF-8 guarantees that any tokenizer can decode any text
   - (d) it does not really round-trip; `errors="replace"` hid the damage

3. `merge([7, 7, 7, 7], (7, 7), 300)` must return:
   - (a) `[300, 300]`  - (b) `[300, 7, 7]`  - (c) `[300, 300, 300, 7]`  - (d) `[300, 7]`

4. Devanagari measured far more tokens per character than English. Under a model priced per
   token whose context limit is counted in tokens, that means a Hindi user:
   - (a) pays more per request, waits longer, and fits less of their document in context
   - (b) pays the same, because pricing is per request
   - (c) pays more but gets a proportionally larger context window to compensate
   - (d) is unaffected, because the model reads characters

5. Retraining the identical code on a balanced corpus cut the penalty sharply. The honest
   conclusion is:
   - (a) byte pair encoding is biased against non-Latin scripts by design
   - (b) the algorithm is indifferent to script; the penalty comes from which text was chosen
         for training, and vocabulary slots spent on one language are not available to another
   - (c) the fix is a larger vocabulary, whatever the corpus contains
   - (d) the first measurement must have been wrong

Answers, with reasoning, are in this lesson's worked solution in the course repository.

## What you built, and where it goes next

A byte-level BPE trainer, an encoder, a decoder that round-trips five scripts it was never
trained on, and a measurement of what the same sentence costs each of them. The merge list
you produced is a testable object: `T03-L02-bpe-merge-loop-in-cpp` rebuilds this loop in C++
and checks its merges against yours, which is why the tie-break and the boundary rule were
specified here rather than left to taste.

In [ ]:
# Your progress board. Every check is re-run here, quietly, against your code as it stands
# now — each one already printed its feedback in its own cell above — so the board is
# current even if you edited an exercise and did not re-run its check.
if __name__ == "__main__":
    _ALL_CHECKS = (  # (exercise, its check, the exercises that check relies on)
        ("exercise 1", _check_get_stats, ()),
        ("exercise 2", _check_merge, ()),
        ("exercise 3", _check_best_pair, ()),
        ("exercise 4", _check_train_bpe, ("exercise 1", "exercise 2", "exercise 3")),
        ("exercise 5", _check_encode_decode, ("exercise 2", "exercise 4")),
        ("exercise 6", _check_token_tax, ("exercise 4", "exercise 5")),
    )
    with contextlib.redirect_stdout(io.StringIO()):
        for _name, _check, _needs in _ALL_CHECKS:
            _try(_name, _check, needs=_needs)
    print(f"lesson wall time so far: {time.perf_counter() - _LESSON_T0:.1f}s\n")
    _progress_board()
    # A stub you have not reached yet is not a failure. A check that ran and came back wrong
    # is: in a script or under CI it ends the run non-zero, rather than letting a green exit
    # code paper over it. Inside a notebook kernel the board above has already said so, in a
    # line rather than a traceback at the foot of the page.
    if _FAILED_CHECKS and "ipykernel" not in sys.modules:
        raise SystemExit("checks failed: " + ", ".join(dict.fromkeys(_FAILED_CHECKS)))